# Synthetic Compact PEST Results Review

Use this notebook after a synthetic compact PEST run finishes. It reopens the run through the built-in `myflopy` API, loads the saved targets, and shows the main review surfaces: summary metadata, residual improvement, `K` changes, and example time-series plots.

## Imports And Display Settings

This cell imports the `myflopy` review API and a few display helpers. The notebook itself does not rebuild or rerun the model.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import myflopy as mf

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## Choose A Completed Run

By default, this looks for the latest synthetic compact demo run using the `*_latest.txt` pointer written by the workflow. You can also point `run_dir` directly at a specific artifact root, `pest/`, or `pest_master/` folder.

In [ ]:
artifact_root = Path(r"C:\Users\lukem\Python\Projects\myflopy\examples\mf6\artifacts")
run_family = "synthetic_compact_pest_demo"
latest_pointer = artifact_root / f"{run_family}_latest.txt"

if latest_pointer.exists():
    run_dir = Path(latest_pointer.read_text(encoding="utf-8").strip())
else:
    run_dir = artifact_root / run_family

run_dir

## Open The Completed Run

This uses the persisted workspace metadata to reopen the baseline model, calibrated model, and saved head targets. `review()` also assembles the common residual and `K` summaries in one object.

In [ ]:
pest_run = mf.open_pest_run(run_dir)
targets = pest_run.load_head_targets()
review = pest_run.review(targets)

baseline_model = review.baseline_model
completed_model = review.calibrated_model

print("Run root:", pest_run.root)
print("Baseline workspace:", pest_run.baseline_workspace)
print("PEST workspace:", pest_run.pest_workspace)
print("Control file:", pest_run.pst_path)
print("Final parameter file:", pest_run.par_path)
print("Saved head-target rows:", len(targets.to_long()))


## Review Summary

Start here to see whether calibration improved the fit overall. The residual comparison table can also be sorted to find wells or periods that improved the most, or the least.

In [ ]:
display(pest_run.summary())
display(review.stats)

residual_view = review.residual_compare.sort_values(
    ["abs_residual_improvement", "time", "name"],
    ascending=[False, True, True],
).reset_index(drop=True)

display(residual_view.head(20))


## K-Field Change

These plots show the calibrated `K` field and the ratio `K_final / K_initial`. The ratio plot is usually the quickest way to see where PEST moved the model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
pest_run.plot_k(ax=axes[0])
pest_run.plot_k_ratio(ax=axes[1])
plt.show()

display(review.k_geodata.describe())


## Truth vs Calibrated K

If the workflow wrote the optional review artifacts, this cell compares the calibrated `K` field to the synthetic truth field directly. That makes it easy to see whether calibration only improved fit, or actually recovered the intended structure.

In [ ]:
review_dir = pest_run.root / "review"
truth_k_path = review_dir / "truth_k.gpkg"
k_review_path = review_dir / "k_review.gpkg"

if truth_k_path.exists() and k_review_path.exists():
    truth_k = gpd.read_file(truth_k_path)
    k_review = gpd.read_file(k_review_path)
    k_compare = k_review.merge(truth_k[["cell", "k_truth"]], on="cell", how="left")
    k_compare["k_final_to_truth"] = k_compare["k_final"] / k_compare["k_truth"]
    display(k_compare[["cell", "k_initial", "k_final", "k_truth", "k_ratio", "k_final_to_truth"]].head(20))

    fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
    k_compare.plot(column="k_truth", ax=axes[0], legend=True)
    axes[0].set_title("Synthetic truth K")
    axes[0].set_axis_off()

    k_compare.plot(column="k_final", ax=axes[1], legend=True)
    axes[1].set_title("K final")
    axes[1].set_axis_off()

    k_compare.plot(column="k_initial", ax=axes[2], legend=True)
    axes[2].set_title("K initial")
    axes[2].set_axis_off()

    plt.show()
else:
    print("Truth-vs-calibrated K artifacts were not found in the review folder.")


## Drain Conductance Review

The synthetic workflow writes a drain summary table so you can see how much each drain feature moved relative to the biased starting conductances.

In [ ]:
drain_summary_path = review_dir / "drain_conductance_summary.csv"

if drain_summary_path.exists():
    drain_summary = pd.read_csv(drain_summary_path)
    drain_summary["relative_change"] = (
        (drain_summary["final_cond"] - drain_summary["start_cond"]).abs()
        / drain_summary["start_cond"]
    )
    display(drain_summary)
else:
    print("Drain conductance summary was not found in the review folder.")


## Period And Well Diagnostics

These plots help answer the usual calibration questions: which periods improved, and what happened at a specific observation point through time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
pest_run.plot_residuals_by_period(targets=targets).show(

)
pest_run.plot_obs_vs_sim(targets=targets).show()
plt.show()

"""well_name = residual_view.loc[0, "name"] if not residual_view.empty else targets.to_long()["name"].iloc[0]
fig, ax = plt.subplots(figsize=(10, 4))
pest_run.plot_well_timeseries(well_name, targets=targets)
ax.set_title(f"Observed vs simulated heads: {well_name}")
plt.show()"""
